In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'/home/sharad/Projects/end_to_end_ml_project_with_mlflow'

In [4]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/sharadvyas123/end_to_end_ml_project_with_mlflow.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "sharadvyas123"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "f01ccc75b8538f2657872391ab2afb7ef0bd7126"

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir : Path
    test_data_path: Path
    model_path : Path
    all_params : dict
    metric_file_name : Path
    target_column : str
    mlflow_uri: str

In [7]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml , create_directories , save_json

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)


        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self):
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN


        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri=os.getenv("MLFLOW_TRACKING_URI")
        )

        return model_evaluation_config

    

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error , mean_absolute_error , r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn 

In [10]:
class ModelEvluation:
    def __init__(self ,config : ModelEvaluationConfig):
        self.config = config


    def eval_metrics(self , actual , pred):
        rmse = np.sqrt(mean_squared_error(actual , pred))
        mae = mean_absolute_error(actual , pred)

        r2 = r2_score(actual ,pred)
        return rmse , mae , r2


    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)


        test_x = test_data.drop([self.config.target_column] , axis= 1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)

        tracking_uri_type_store = urlparse(mlflow.get_registry_uri()).scheme


        with mlflow.start_run():
            predict_qualities = model.predict(test_x)

            (rmse , mae , r2) = self.eval_metrics(test_y , predict_qualities)


            scores = {"rmse" : rmse , "mae" : mae , "r2" : r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric('rmse' , rmse)
            mlflow.log_metric('mae' , mae)
            mlflow.log_metric('r2' , r2)


            if tracking_uri_type_store != "file":
                mlflow.sklearn.log_model(model , "model" ,registered_model_name= "ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model , "model")

In [11]:
try:
    config= ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()

    model_evaluation = ModelEvluation(config=model_evaluation_config)

    model_evaluation.log_into_mlflow()
    
except Exception as e:
    raise e

[2026-11-12 19:29:26,369 : INFO : common : yaml file: config/config.yaml loaded successfully]
[2026-11-12 19:29:26,372 : INFO : common : yaml file: params.yaml loaded successfully]
[2026-11-12 19:29:26,375 : INFO : common : yaml file: schema.yaml loaded successfully]
[2026-11-12 19:29:26,376 : INFO : common : created directory at: artifacts]
[2026-11-12 19:29:26,377 : INFO : common : created directory at: artifacts/model_evaluation]
[2026-11-12 19:29:27,769 : INFO : common : json file saved at: artifacts/model_evaluation/metrics.json]


/home/sharad/Projects/end_to_end_ml_project_with_mlflow/venv/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/11/12 19:29:34 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticnetModel, version 2
Created version '2' of model 'ElasticnetModel'.
